In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pickle

pd.set_option("display.max_columns", None)

print("Libraries Loaded")

Libraries Loaded


In [4]:
video_df = pd.read_csv(
    r"E:\Reel-Recommendation-Engine\data\processed\videos_cleaned.csv"
)

print(video_df.shape)

video_df.head()

(7583, 12)


,video_id,author_id,video_type,upload_dt,upload_type,visible_status,video_duration,server_width,server_height,music_id,music_type,tag
0,0,7349781,NORMAL,2022-04-10,LongImport,0.0,87433.0,720.0,1280.0,9155697141,9.0,39
1,1,2103883,NORMAL,2022-04-10,Kmovie,0.0,218066.0,720.0,1280.0,6355810746,9.0,2
2,2,5067285,NORMAL,2022-04-09,ShortImport,0.0,9233.0,720.0,1280.0,6618412736,4.0,1
3,3,7048760,NORMAL,2022-04-11,Web,0.0,16433.0,720.0,1280.0,9161677205,9.0,7
4,4,8635271,NORMAL,2022-04-09,Web,0.0,38766.0,720.0,1280.0,9141092381,9.0,9


In [5]:
video_df[
    [
        "video_type",
        "tag",
        "author_id",
        "music_id",
        "upload_type"
    ]
].isnull().sum()

video_type     0
tag            0
author_id      0
music_id       0
upload_type    0
dtype: int64

In [6]:
video_df["tag"] = video_df["tag"].astype(str)

video_df["author_id"] = video_df["author_id"].astype(str)

video_df["music_id"] = video_df["music_id"].astype(str)

video_df["video_type"] = video_df["video_type"].astype(str)

video_df["upload_type"] = video_df["upload_type"].astype(str)

In [7]:
video_df["content_features"] = (

      video_df["video_type"]

    + " "

    + video_df["tag"]

    + " "

    + video_df["author_id"]

    + " "

    + video_df["music_id"]

    + " "

    + video_df["upload_type"]

)

In [8]:
video_df[
    [
        "video_id",
        "content_features"
    ]
].head()

,video_id,content_features
0,0,NORMAL 39 7349781 9155697141 LongImport
1,1,NORMAL 2 2103883 6355810746 Kmovie
2,2,NORMAL 1 5067285 6618412736 ShortImport
3,3,NORMAL 7 7048760 9161677205 Web
4,4,NORMAL 9 8635271 9141092381 Web


In [9]:
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(
    video_df["content_features"]
)

print(tfidf_matrix.shape)

(7583, 13763)


In [10]:
cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print(cosine_sim.shape)

(7583, 7583)


In [11]:
video_indices = pd.Series(
    video_df.index,
    index=video_df["video_id"]
).drop_duplicates()

video_indices.head()

video_id
0    0
1    1
2    2
3    3
4    4
dtype: int64

In [12]:
def recommend_similar_videos(
        video_id,
        top_n=10):

    idx = video_indices.get(video_id)

    if idx is None:
        return "Video Not Found"

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    video_indexes = [
        i[0]
        for i in similarity_scores
    ]

    return video_df[
        [
            "video_id",
            "tag",
            "video_type"
        ]
    ].iloc[video_indexes]

In [13]:
sample_video = video_df["video_id"].iloc[0]

print("Video ID:", sample_video)

recommend_similar_videos(sample_video)

Video ID: 0


,video_id,tag,video_type
4121,4121,39,UNKNOWN
655,655,39,NORMAL
3234,3234,39,NORMAL
2894,2894,39,NORMAL
481,481,"39,68",NORMAL
3174,3174,"39,68",NORMAL
817,817,39,NORMAL
103,103,39,NORMAL
892,892,39,NORMAL
935,935,39,NORMAL


In [14]:
import os

os.makedirs(
    r"E:\Reel-Recommendation-Engine\models",
    exist_ok=True
)

In [15]:
with open(
    r"E:\Reel-Recommendation-Engine\models\tfidf_vectorizer.pkl",
    "wb"
) as f:

    pickle.dump(tfidf, f)

In [16]:
with open(
    r"E:\Reel-Recommendation-Engine\models\content_similarity.pkl",
    "wb"
) as f:

    pickle.dump(cosine_sim, f)

In [17]:
video_df.to_csv(
    r"E:\Reel-Recommendation-Engine\models\content_metadata.csv",
    index=False
)

print("Content-Based Models Saved Successfully")

Content-Based Models Saved Successfully


In [18]:
import pickle

with open(
    r"E:\Reel-Recommendation-Engine\models\video_indices.pkl",
    "wb"
) as f:

    pickle.dump(video_indices, f)

print("Video Index Mapping Saved")

Video Index Mapping Saved
